# 03 — Validation Protocol, Model Comparison, Reconciliation & Capacity Policy

This notebook presents the rolling-origin cross-validation results, the 24-step holdout benchmark, top-down vs bottom-up hierarchical reconciliation, conformalized empirical prediction intervals, and the virtual-core capacity headroom policy evaluation.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

TABLES_DIR = Path('../reports/tables')
FIGS_DIR = Path('../reports/figures')

cv_df = pd.read_csv(TABLES_DIR / 'aggregate_cv_metrics.csv')
ho_df = pd.read_csv(TABLES_DIR / 'aggregate_holdout_metrics.csv')
hz_df = pd.read_csv(TABLES_DIR / 'horizon_decay_metrics.csv')
rec_df = pd.read_csv(TABLES_DIR / 'reconciliation_comparison.csv')
cov_df = pd.read_csv(TABLES_DIR / 'interval_coverage.csv')
pol_df = pd.read_csv(TABLES_DIR / 'interval_coverage_and_policy.csv')

## 1. Rolling-Origin Cross-Validation (4 Folds on 110 Training Timestamps) vs 24-Step Holdout
We compare all 5 baselines (`Persistence`, `Seasonal Naive`, `MA(12)`, `Drift`, `EWMA`) and 4 candidate models (`Damped ETS`, `Direct Ridge`, `Direct HGB`, `Hybrid Ensemble`).

In [2]:
merged = cv_df.merge(ho_df, on=['target', 'model', 'model_label', 'family'])
for t in ['fleet_avg_cpu', 'fleet_p95_max_cpu', 'fleet_busy_vm_fraction']:
    print(f'=== Target: {t} ===')
    sub = merged[merged['target'] == t][[
        'model_label', 'family', 'cv_mae_mean', 'cv_rmse_mean', 'holdout_mae', 'holdout_rmse', 'holdout_nmae', 'holdout_mase'
    ]].sort_values('holdout_mae')
    display(sub.round(4))

=== Target: fleet_avg_cpu ===


,model_label,family,cv_mae_mean,cv_rmse_mean,holdout_mae,holdout_rmse,holdout_nmae,holdout_mase
8,Hybrid Ensemble,Candidate,0.2027,0.2362,0.1061,0.1423,0.7229,0.8761
7,Direct HGB,Candidate,0.2842,0.3188,0.1195,0.1408,0.8142,0.9867
2,MA(12),Baseline,0.2468,0.2764,0.1230,0.1613,0.8380,1.0156
4,EWMA,Baseline,0.2553,0.2906,0.1392,0.1770,0.9485,1.1495
6,Direct Ridge,Candidate,0.3499,0.3969,0.1455,0.1663,0.9912,1.2012
1,Seasonal Naive,Baseline,0.3111,0.3517,0.1464,0.1671,0.9974,1.2087
0,Persistence,Baseline,0.2445,0.2807,0.1468,0.1848,1.0000,1.2119
5,Damped ETS,Candidate,0.2847,0.3224,0.1567,0.1941,1.0672,1.2934
3,Drift,Baseline,0.2780,0.3140,0.1820,0.2206,1.2402,1.5030


=== Target: fleet_p95_max_cpu ===


,model_label,family,cv_mae_mean,cv_rmse_mean,holdout_mae,holdout_rmse,holdout_nmae,holdout_mase
17,Hybrid Ensemble,Candidate,3.6797,4.4971,0.6788,0.8999,0.8209,0.6076
11,MA(12),Baseline,3.3168,3.8030,0.7269,0.9449,0.8790,0.6506
13,EWMA,Baseline,3.1914,3.6964,0.7816,0.9852,0.9452,0.6996
14,Damped ETS,Candidate,3.2903,3.8436,0.8005,1.0014,0.9680,0.7166
9,Persistence,Baseline,2.9144,3.4377,0.8270,1.0247,1.0000,0.7402
16,Direct HGB,Candidate,2.8034,3.3466,0.9506,1.1387,1.1495,0.8509
12,Drift,Baseline,3.6007,4.1517,1.0051,1.2028,1.2154,0.8997
10,Seasonal Naive,Baseline,4.6752,5.3936,1.0441,1.2436,1.2626,0.9346
15,Direct Ridge,Candidate,11.9585,15.7953,2.1108,2.3820,2.5525,1.8895


=== Target: fleet_busy_vm_fraction ===


,model_label,family,cv_mae_mean,cv_rmse_mean,holdout_mae,holdout_rmse,holdout_nmae,holdout_mase
19,Seasonal Naive,Baseline,0.0038,0.0049,0.0038,0.0049,0.9611,1.3992
18,Persistence,Baseline,0.0051,0.0061,0.0040,0.0042,1.0000,1.4558
26,Hybrid Ensemble,Candidate,0.0043,0.0051,0.0040,0.0043,1.0116,1.4727
23,Damped ETS,Candidate,0.0055,0.0064,0.0040,0.0043,1.0152,1.4779
22,EWMA,Baseline,0.0043,0.0052,0.0040,0.0043,1.0170,1.4806
20,MA(12),Baseline,0.0038,0.0044,0.0041,0.0044,1.0233,1.4897
25,Direct HGB,Candidate,0.0046,0.0056,0.0041,0.0047,1.0385,1.5118
21,Drift,Baseline,0.0064,0.0073,0.0042,0.0044,1.0474,1.5248
24,Direct Ridge,Candidate,0.0051,0.0063,0.0043,0.0050,1.0818,1.5748


## 2. Top-Down Aggregate vs Bottom-Up Stratum & VM-Level Shrinkage Reconciliation
Comparing fleet-level direct models against bottom-up stratum panel models and 10,000-VM empirical-Bayes shrinkage.

In [3]:
display(rec_df.sort_values('holdout_mae').round(4))

,approach,paradigm,holdout_mae,holdout_rmse,holdout_smape,holdout_nmae,holdout_mase
3,"Bottom-Up Stratum Panel (Ridge, Category x Cor...",Segment Bottom-Up,0.0802,0.0980,0.8646,0.5462,0.6620
6,Global VM Panel Blend (50% Stratum Ridge + 50%...,Hierarchical Reconciliation,0.0934,0.1150,1.0067,0.6361,0.7709
2,Top-Down Aggregate (Hybrid Ensemble),Aggregate Top-Down,0.1061,0.1423,1.1430,0.7229,0.8761
5,"VM-Level Empirical-Bayes Shrinkage (10,000 VMs)",VM-Level Bottom-Up,0.1066,0.1343,1.1488,0.7260,0.8799
4,"Bottom-Up Stratum Panel (HGB, Category x Core ...",Segment Bottom-Up,0.1243,0.1515,1.3396,0.8466,1.0260
1,Top-Down Aggregate (Direct Ridge),Aggregate Top-Down,0.1455,0.1663,1.5604,0.9912,1.2012
0,Top-Down Aggregate (Persistence Baseline),Aggregate Top-Down,0.1468,0.1848,1.5851,1.0000,1.2119


## 3. Error Decay by Horizon Bucket (`h = 1..6`, `7..12`, `13..24`)

In [4]:
display(hz_df[hz_df['target'] == 'fleet_avg_cpu'].pivot(index='model_label', columns='horizon_bucket', values='mae').round(4))

horizon_bucket,"Full Holdout (h=1..24, 5-120m)","Long (h=13..24, 65-120m)","Medium (h=7..12, 35-60m)","Short (h=1..6, 5-30m)"
model_label,,,,
Damped ETS,0.1567,0.2210,0.0848,0.0997
Direct HGB,0.1195,0.1083,0.1171,0.1442
Direct Ridge,0.1455,0.1505,0.1606,0.1204
Hybrid Ensemble,0.1061,0.1365,0.0426,0.1089
MA(12),0.1230,0.1543,0.0468,0.1366
Persistence,0.1468,0.2060,0.0717,0.1034


## 4. Empirical Prediction Interval Coverage & vCore Capacity Headroom Policy
Using out-of-fold residuals with neighbor pooling and conformal finite-sample correction, our `Horizon-Adaptive P90 Upper Bound` policy eliminates SLA shortfalls across the 28,388-vCore fleet while minimizing asymmetric operational loss.

In [5]:
display(cov_df.round(4))
display(pol_df[pol_df['scenario'] == 'Standard 5-Min Mean Demand'][[
    'policy', 'mean_provisioned_pct', 'mean_provisioned_cores', 'sla_violation_rate_pct',
    'mean_shortfall_pct', 'mean_overprovision_pct', 'asymmetric_cost', 'is_recommended'
]].round(3))

,target,bound,nominal,empirical_coverage,mean_width,pinball_loss
0,fleet_avg_cpu,p80_upper,0.80,0.9583,0.3258,0.0552
1,fleet_avg_cpu,p90_upper,0.90,1.0000,0.4621,0.0411
2,fleet_avg_cpu,p95_upper,0.95,1.0000,0.5430,0.0246
3,fleet_avg_cpu,pi80,0.80,1.0000,0.8141,NaN
4,fleet_avg_cpu,pi90,0.90,1.0000,1.0318,NaN
5,fleet_p95_max_cpu,p80_upper,0.80,1.0000,6.1370,1.1687
6,fleet_p95_max_cpu,p90_upper,0.90,1.0000,8.1364,0.7843
7,fleet_p95_max_cpu,p95_upper,0.95,1.0000,11.2759,0.5491
8,fleet_p95_max_cpu,pi80,0.80,1.0000,14.9120,NaN
9,fleet_p95_max_cpu,pi90,0.90,1.0000,21.5398,NaN


,policy,mean_provisioned_pct,mean_provisioned_cores,sla_violation_rate_pct,mean_shortfall_pct,mean_overprovision_pct,asymmetric_cost,is_recommended
0,Point Forecast (No Headroom),9.217,2617.379,62.500,0.079,0.027,0.421,False
1,Static Safety Margin (+5% relative),9.678,2748.248,0.000,0.000,0.409,0.409,False
2,Static Safety Margin (+10% relative),10.139,2879.117,0.000,0.000,0.870,0.870,False
3,Horizon-Adaptive P80 Upper Bound,9.543,2709.899,4.167,0.000,0.275,0.276,False
4,Horizon-Adaptive P90 Upper Bound (Recommended),9.679,2748.588,0.000,0.000,0.411,0.411,True
5,Horizon-Adaptive P95 Upper Bound (Conservative),9.760,2771.576,0.000,0.000,0.491,0.491,False
